# W15 — Explainability Figures and Maintenance Decision Scenarios

**Purpose.** Produce the material of Sections 5.6–5.7 of the article that W12 does not: explanation figures for individual engines, and the link from prediction to maintenance recommendation.

1. **Models.** One cross-validation fold (seed 42, fold 0) of W12 is retrained for three configurations, with the W13 hyperparameters and the same pipeline (selection, rate-aware ML representation, BiLSTM with collapse guard, α fusion): **S** (sensors), **S+T** (sensors + operating age: the recommended predictive configuration) and **S+T+Op+A** (all enterprise data: used for the explanation figures, so that every group of variables appears).
2. **Uncertainty.** Split-conformal residuals of the hybrid, by predicted life stage, from the inner-validation engines of the fold (the test engines are never used for calibration). They give 90% prediction intervals and the failure risk P(RUL ≤ h). Coverage is checked on the test engines.
3. **Trajectories.** For two test engines (one FD001, one FD003), the hybrid prediction at every cycle with its interval.
4. **Explanations.** SHAP (XGBoost branch): summary, waterfall at a decision point, contribution of each group of variables along the trajectory. Attention (BiLSTM): feature- and temporal-attention weights at the same decision point, and their evolution.
5. **Decisions.** The scenario generator of `src/scenario_generator.py`, with a **risk-aware** cost model: the failure probability of a preventive-maintenance (PM) scenario at cycle t is the model's P(RUL < t − now), and its expected cost is (1 − p)·PM cost + p·corrective cost. For two decision points (an engine in mid-life, an engine near failure), four policies are compared:
   - **full**: S+T prediction, conformal uncertainty, actual enterprise context at that cycle;
   - **no context**: same prediction and uncertainty, enterprise context replaced by typical fleet values;
   - **point estimate**: actual context, but the RUL is taken as certain (no uncertainty);
   - **sensors only**: S prediction instead of S+T, with uncertainty and actual context.
6. **Fleet-level decision evaluation.** The four policies are applied at every truncation point of the 40 test engines of the fold, and their outcomes are scored against the true failure time: share of failures before the planned PM, unused life, realised cost (actual context).

Results go to a dedicated, timestamped folder on Drive (`PhD_results/W15_...`).

## 0. Setup and data (same code as W12)

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path
from datetime import datetime

IN_COLAB = 'COLAB_GPU' in os.environ or os.path.exists('/content')

if IN_COLAB:
    # Fresh clone every run; cache and results live on Drive so finished work survives a disconnect
    os.chdir('/content')     # leave the old clone before deleting it (a re-run starts inside it)
    if os.path.exists('/content/PhD-Project-'):
        !rm -rf /content/PhD-Project-
    !git clone https://github.com/f-khadija-benzine/PhD-Project-.git /content/PhD-Project-
    !pip install xgboost shap -q
    os.chdir('/content/PhD-Project-/src')
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/PhD-Project-')
    TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
    SAVE_DIR = Path(f'/content/drive/MyDrive/PhD_results/W15_{TIMESTAMP}')
    CACHE_ROOT = Path('/content/drive/MyDrive/PhD_results/W15_cache')
else:
    PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
    SAVE_DIR = None
    CACHE_ROOT = PROJECT_ROOT / 'results' / 'W15_cache'

sys.path.insert(0, str(PROJECT_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_ROOT))

CMAPSS_DIR = PROJECT_ROOT / 'data' / 'raw_data' / 'C_MAPSS'
BI_DIR     = PROJECT_ROOT / 'data' / 'raw_data' / 'BI_data'
ASSET_DIR  = BI_DIR / 'asset'
# ---- Fleet: FD001 alone, or FD001 + FD003 analysed as ONE fleet of 200 engines ----
# FD003: same single operating condition as FD001, two fault modes (HPC + fan).
# Unit ids of FD003 are offset by +1000. Set DATASETS = ['FD001'] to reproduce the FD001-only runs.
DATASETS = ['FD001', 'FD003']
for _ds in DATASETS:
    assert (CMAPSS_DIR / f'train_{_ds}.txt').exists(), f"train_{_ds}.txt missing in {CMAPSS_DIR}: upload it to the repo"
FLEET_TAG = '_'.join(DATASETS)
TAG_SUFFIX = '' if FLEET_TAG == 'FD001' else f'_{FLEET_TAG}'   # FD001 keeps its original paths and caches

RESULTS    = PROJECT_ROOT / 'results' / f'W15{TAG_SUFFIX}'
CACHE_ROOT = CACHE_ROOT.parent / f'{CACHE_ROOT.name}{TAG_SUFFIX}'
if SAVE_DIR is not None:
    SAVE_DIR = SAVE_DIR.parent / f'{SAVE_DIR.name}{TAG_SUFFIX}'
for d in (RESULTS, CACHE_ROOT):
    d.mkdir(parents=True, exist_ok=True)
if SAVE_DIR is not None:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Fleet: {FLEET_TAG}\nProject root: {PROJECT_ROOT}\nResults: {RESULTS}\nCache: {CACHE_ROOT}\nDrive save: {SAVE_DIR}")

In [ ]:
# Regenerate both BI blocks from the generators in the repo, so data and code always match
for script in ['generate_bi_data.py', 'generate_bi_asset_data.py']:
    res = subprocess.run([sys.executable, str(PROJECT_ROOT / 'src' / script)],
                         capture_output=True, text=True, cwd=str(PROJECT_ROOT))
    print(f"--- {script}: {'OK' if res.returncode == 0 else 'FAILED'}")
    if res.returncode != 0:
        print(res.stderr[-2000:])
        raise RuntimeError(f"{script} failed")

In [ ]:
import importlib, time, json, io, contextlib, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from numpy.lib.stride_tricks import sliding_window_view
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import shap

import ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, generate_bi_asset_data as gba
for m in (ml_branch, bi_fusion, attention, feature_selection, feature_selection_aficv, gba):
    importlib.reload(m)
from ml_branch import MLBranch
from attention import build_dual_attention_bilstm
from feature_selection import BIAwareFeatureSelector
from feature_selection_aficv import AFICvFeatureSelector
from bi_fusion import CATEGORICAL_BI_VARS, CONTINUOUS_BI_VARS, BI_FEATURE_NAMES

# ---------------- Experiment configuration (fixed before running) ----------------
RHO = 0.4                      # asset-history link strength, chosen from W11 realism checks
FLEET = 2026                   # reference simulated fleet (= repo files)
SEEDS = [42, 43, 44]           # repeated CV (advisor point 4)
W = 30
RUL_MAX = 125
N_FOLDS = 5
N_CUTS = 3
INNER_VAL_FRAC = 0.2
CUT_SEED = 42                  # same truncation points as W11
ASSET_ROUTING = 'ml_only'      # static asset history goes to the ML branch only (see Section 4.1)

RF_PARAMS = dict(n_estimators=300, max_depth=15, min_samples_split=5, min_samples_leaf=2)   # benchmark, not tuned
BATCH, EPOCHS = 128, 100
DENSE_EPOCHS = 300           # dense fusion network (small, early-stopped; independent of QUICK)
CACHE_VERSION = 'v3b'          # v3 = rate-aware ML representation + operating age; b = BiLSTM collapse guard
EVAL_MIN_HISTORY = W          # score only truncation points with ≥ W observed cycles (see Stage 1)
MAX_DL_RETRIES = 2             # BiLSTM retrained with a new initialisation if it collapses to a constant (see train_dl)
REP = 'rate'                   # ML branch: 'rate' = sensors -> window statistics, slow BI and age -> last value, static -> once
                               #            'stats' = window statistics for every time-varying column (v2 design)
RUN_ROUTING_CHECK = False      # Section 4.1 (asset history to both branches); shown in v2, set True to repeat it
CORR_SEL = dict(variance_threshold=0.01, correlation_threshold=0.95)
AFICV_SEL = dict(base_learner='xgboost', n_folds=5, cumulative_threshold=0.90)

# QUICK = True: smoke test (tiny models, 1 seed). Use False for reported results.
QUICK = False
if QUICK:
    SEEDS = [42]; EPOCHS = 2
    RF_PARAMS = dict(n_estimators=30, max_depth=8, min_samples_split=5, min_samples_leaf=2)

GPU = tf.config.list_physical_devices('GPU')
XGB_EXTRA = {'device': 'cuda', 'tree_method': 'hist'} if GPU else {}
CACHE_DIR = CACHE_ROOT / ('quick' if QUICK else 'full')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print(f"QUICK={QUICK} | seeds={SEEDS} | epochs={EPOCHS} | GPU={bool(GPU)} | cache={CACHE_DIR}")

In [ ]:
cols = ['unit', 'cycle'] + [f'setting_{i}' for i in range(1, 4)] + [f'sensor_{i}' for i in range(1, 22)]
def _load_train(ds):
    t = pd.read_csv(CMAPSS_DIR / f'train_{ds}.txt', sep=r'\s+', header=None, names=cols, engine='python')
    t['unit'] += gba.UNIT_OFFSET[ds]
    t['subset'] = ds
    return t
train = pd.concat([_load_train(ds) for ds in DATASETS], ignore_index=True)
train['total_life'] = train.groupby('unit')['cycle'].transform('max')
train['rul'] = (train['total_life'] - train['cycle']).clip(upper=RUL_MAX)
UNITS = np.sort(train['unit'].unique())
LIFE = train.groupby('unit')['total_life'].first()

SENSOR_COLS = [c for c in train.columns if c.startswith('sensor_')]
SETTING_COLS = [c for c in train.columns if c.startswith('setting_')]

# Operational BI (13 variables)
bi_op = pd.read_csv(BI_DIR / f'BI_{FLEET_TAG}_train.csv').rename(columns={'unit_id': 'unit'})
OP_VARS = [c for c in BI_FEATURE_NAMES if c in bi_op.columns]
OP_CAT = [c for c in CATEGORICAL_BI_VARS if c in OP_VARS]
OP_CONT = [c for c in CONTINUOUS_BI_VARS if c in OP_VARS]

# Asset-history BI (5 variables), reference fleet, rho fixed above
asset = pd.read_csv(ASSET_DIR / f"BI_{FLEET_TAG}_asset_{gba.rho_tag(RHO)}.csv").rename(columns={'unit_id': 'unit'})
ASSET_VARS = gba.ASSET_VARS
ASSET_CONT = ['prior_unplanned_removals', 'hours_since_overhaul_entry']
BATCH_LEVELS = [f"B{i}" for i in range(1, gba.N_BATCHES + 1)]

def build_table(op_df, asset_df):
    # One table with sensors + all BI, categorical variables one-hot encoded
    df = train[['unit', 'cycle', 'rul'] + SETTING_COLS + SENSOR_COLS].copy()
    df['cycles_in_service'] = df['cycle'].astype(float)      # operating age: CMMS usage counter, known online
    df = df.merge(
        op_df[['unit', 'cycle'] + OP_VARS], on=['unit', 'cycle'], how='left')
    for c in OP_CAT:
        df[c] = pd.Categorical(df[c].astype(int), categories=sorted(bi_op[c].unique()))
    df = pd.get_dummies(df, columns=OP_CAT, dtype=float)
    a = asset_df[['unit'] + ASSET_VARS].copy()
    a['component_batch'] = pd.Categorical(a['component_batch'], categories=BATCH_LEVELS)
    df = df.merge(pd.get_dummies(a, columns=['component_batch'], dtype=float), on='unit', how='left')
    return df.sort_values(['unit', 'cycle']).reset_index(drop=True)

# Falsification controls (same methods as W10/W11):
#  - operational BI: each variable permuted independently across all rows
#  - asset history: full records permuted across engines (stays constant within an engine)
op_shuf = bi_op.copy()
r_op = np.random.RandomState(42)
for c in OP_VARS:
    op_shuf[c] = r_op.permutation(op_shuf[c].values)
asset_shuf = asset.sort_values('unit').reset_index(drop=True).copy()
perm = np.random.default_rng(FLEET + 1000).permutation(len(asset_shuf))
asset_shuf[ASSET_VARS] = asset_shuf[ASSET_VARS].values[perm]

DATA = {'real': build_table(bi_op, asset),
        'asset_shuffled': build_table(bi_op, asset_shuf),
        'all_shuffled': build_table(op_shuf, asset_shuf)}
DF = DATA['real']
assert all(list(d.columns) == list(DF.columns) for d in DATA.values())

OP_COLS = [c for c in DF.columns if c in OP_CONT or c in OP_VARS or c.startswith(tuple(f"{v}_" for v in OP_CAT))]
ASSET_COLS = [c for c in DF.columns if c.startswith(tuple(ASSET_VARS))]
AGE_COLS = ['cycles_in_service']
# Feature sets chosen after W11 (Section 10): operating age and rate-aware BI. T = time in service (operating age).
FEATURE_SETS = {
    'S':          {'bi': [],                               'data': 'real'},          # sensors + settings only
    'S+T':        {'bi': AGE_COLS,                         'data': 'real'},          # + operating age (CMMS)
    'S+Op+A':     {'bi': OP_COLS + ASSET_COLS,             'data': 'real'},          # + 18 BI (13 operational + 5 asset history)
    'S+T+Op+A':   {'bi': AGE_COLS + OP_COLS + ASSET_COLS,  'data': 'real'},          # all enterprise data
    'S+T+Op~+A~': {'bi': AGE_COLS + OP_COLS + ASSET_COLS,  'data': 'all_shuffled'},  # control: age real, 18 BI shuffled
}
MAIN_FSETS = ['S', 'S+T', 'S+Op+A', 'S+T+Op+A']
FULL = 'S+T+Op+A'
CONT_COLS = SENSOR_COLS + SETTING_COLS + OP_CONT + ASSET_CONT + AGE_COLS
RATE_LAST = OP_COLS + AGE_COLS          # columns given to the ML branch by their last known value (REP = 'rate')
GROUP_OF = {**{c: 'sensor' for c in SENSOR_COLS + SETTING_COLS}, **{c: 'operating age' for c in AGE_COLS},
            **{c: 'operational BI' for c in OP_COLS}, **{c: 'asset BI' for c in ASSET_COLS}}

print(f"Fleet {FLEET_TAG}: " + ", ".join(f"{s} {n} engines" for s, n in train.groupby("subset").unit.nunique().items()))
print(f"Engines: {len(UNITS)}, rows: {len(DF)}, life [{LIFE.min()}, {LIFE.max()}]")
print(f"Columns: {len(SENSOR_COLS)} sensors + {len(SETTING_COLS)} settings + {len(OP_COLS)} operational BI "
      f"(13 variables) + {len(ASSET_COLS)} asset BI (5 variables) + operating age")

In [ ]:
BEST_PARAMS_PATH = (Path(f'/content/drive/MyDrive/PhD_results/W13_best_params{TAG_SUFFIX}.json') if IN_COLAB
                    else PROJECT_ROOT / 'results' / f'W13{TAG_SUFFIX}' / f'W13_best_params{TAG_SUFFIX}.json')
DL_FIXED = {'feature_attention_dim': 32, 'dropout_rate': 0.3}
if QUICK:
    _ml = {'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.1, 'flatten': 'statistics'}
    _dl = {'lstm_units': 32, 'temporal_attention_dim': 64, 'dense_units': 32, 'learning_rate': 0.001, **DL_FIXED}
    TUNED = {f: {'ml': _ml, 'dl': _dl} for f in MAIN_FSETS}
else:
    if not BEST_PARAMS_PATH.exists():
        raise FileNotFoundError(f"{BEST_PARAMS_PATH} not found: run W13 (hyperparameter optimisation) first.")
    _b = json.loads(BEST_PARAMS_PATH.read_text())
    if _b.get('quick'):
        raise ValueError("W13 was run with QUICK = True: re-run W13 with QUICK = False.")
    TUNED = _b['params']

def params_for(fset):
    # Shuffled controls use the settings of the 18-BI configuration they are compared with
    return TUNED[fset if fset in TUNED else FULL]

display(pd.DataFrame({f: {**{f"ML {k}": v for k, v in TUNED[f]['ml'].items()},
                          **{f"DL {k}": v for k, v in TUNED[f]['dl'].items()}} for f in MAIN_FSETS}))

In [ ]:
folds = list(GroupKFold(n_splits=N_FOLDS).split(UNITS, groups=UNITS))

rng = np.random.default_rng(CUT_SEED)
cut_rows = []
for u in UNITS:
    life = int(LIFE[u])
    lo, hi = max(int(0.10 * life), 1), max(int(0.95 * life), 2)
    for k, c in enumerate(np.sort(rng.integers(lo, hi, N_CUTS))):
        cut_rows.append({'unit': u, 'cut_id': k, 'cut_cycle': int(c), 'true_rul': min(life - int(c), RUL_MAX)})
CUTS = pd.DataFrame(cut_rows)

def inner_split(train_units, seed):
    r = np.random.default_rng(seed)
    val = r.choice(train_units, size=int(round(INNER_VAL_FRAC * len(train_units))), replace=False)
    return np.setdiff1d(train_units, val), np.sort(val)

assert all(len(set(UNITS[tr]) & set(UNITS[te])) == 0 for tr, te in folds)
print(f"{N_FOLDS} folds × {[len(te) for _, te in folds]} test engines; "
      f"{len(CUTS)} truncation points, true RUL {CUTS.true_rul.min()}–{CUTS.true_rul.max()}")

In [ ]:
STAT_SUFFIX = ('_mean', '_std', '_min', '_max', '_last', '_trend')
def is_static(c): return c.startswith(tuple(ASSET_VARS))
def original_name(flat):
    for s in STAT_SUFFIX:
        if flat.endswith(s):
            return flat[: -len(s)]
    return flat

def quiet(fn, *args, **kw):
    with contextlib.redirect_stdout(io.StringIO()):
        return fn(*args, **kw)

def scale_fold(train_units, fset):
    d = DATA[FEATURE_SETS[fset]['data']].copy()
    sc = MinMaxScaler().fit(d.loc[d.unit.isin(train_units), CONT_COLS])
    d[CONT_COLS] = sc.transform(d[CONT_COLS])
    return d

def select_features(d, train_units, fset, method, exclude_asset=False):
    # Returns the selected column list, fitted on training engines only
    tr = d[d.unit.isin(train_units)]
    bi = FEATURE_SETS[fset]['bi']
    if exclude_asset:
        bi = [c for c in bi if c not in ASSET_COLS]
    if method == 'correlation':
        sel = BIAwareFeatureSelector(**CORR_SEL)
        return quiet(sel.select_features, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                     setting_cols=SETTING_COLS, exclude_cols=['unit', 'cycle', 'rul'])
    if method == 'aficv':
        sel = AFICvFeatureSelector(**AFICV_SEL)
        if bi:
            return quiet(sel.select_features_stratified, data=tr, sensor_cols=SENSOR_COLS, bi_cols=bi,
                         setting_cols=SETTING_COLS, target_col='rul', group_col='unit')
        return quiet(sel.select_features, data=tr, feature_cols=SENSOR_COLS + SETTING_COLS,
                     target_col='rul', group_col='unit')
    raise ValueError(method)

def windows_train(d, units, feats):
    # PAD=False as in the article: windows ending at cycles W..T of each engine
    X, y, owner = [], [], []
    for u in units:
        g = d[d.unit == u]
        v = g[feats].to_numpy(np.float32)
        if len(v) < W:
            continue
        X.append(sliding_window_view(v, (W, v.shape[1]))[:, 0])
        y.append(g['rul'].to_numpy(np.float32)[W - 1:])
        owner += [u] * (len(v) - W + 1)
    return np.concatenate(X), np.concatenate(y), np.asarray(owner)

def windows_cuts(d, units, feats):
    # Last window before each truncation point (zero-padded if the engine is shorter than W)
    X, keys = [], []
    for u in units:
        v_full = d[d.unit == u][feats].to_numpy(np.float32)
        for _, c in CUTS[CUTS.unit == u].iterrows():
            v = v_full[:c.cut_cycle]
            if len(v) < W:
                v = np.vstack([np.zeros((W - len(v), len(feats)), np.float32), v])
            X.append(v[-W:]); keys.append((u, int(c.cut_id)))
    return np.asarray(X), keys

def static_rows(d, owners, static_cols):
    return d.groupby('unit')[static_cols].first().loc[list(owners)].to_numpy(np.float32)

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

def ml_design(ml, X, dyn, d, owners, stat):
    # ML-branch feature matrix. REP = 'rate': per-cycle columns (sensors, settings) -> window representation of
    # MLBranch; slow BI (refreshed every 10-50 cycles) and operating age -> last value in the window; static asset
    # history -> once per sample. REP = 'stats': window representation for every time-varying column.
    last = [i for i, c in enumerate(dyn) if REP == 'rate' and c in RATE_LAST]
    win = [i for i in range(len(dyn)) if i not in last]
    F = ml._flatten(X[:, :, win], [dyn[i] for i in win]); names = list(ml.feature_names_flat)
    if last:
        F = np.hstack([F, X[:, -1, last]]); names += [f"{dyn[i]}_last" for i in last]
    if stat:
        F = np.hstack([F, static_rows(d, owners, stat)]); names += stat
    return F, names

## 1. Train one fold for three configurations

In [ ]:
import scenario_generator as sg
importlib.reload(sg)
from scenario_generator import ScenarioGenerator, UnitContext

SEED, FOLD = 42, 0
CONFIGS = ['S', 'S+T', FULL]
DEC_FSET, BASE_FSET, EXPL_FSET = 'S+T', 'S', FULL
ALPHA_CONF = 0.10
tr_idx, te_idx = folds[FOLD]
TR_U, TE_U = UNITS[tr_idx], UNITS[te_idx]
FIT_U, VAL_U = inner_split(TR_U, SEED * 100 + FOLD)

def fit_fold(fset):
    d = scale_fold(TR_U, fset)
    f_ml = select_features(d, TR_U, fset, 'correlation')
    f_dl = select_features(d, TR_U, fset, 'aficv', exclude_asset=True)
    hp = params_for(fset)
    # ML branch (XGBoost), same representation as W12
    dyn = [c for c in f_ml if not is_static(c)]; stat = [c for c in f_ml if is_static(c)]
    Xtr, ytr, otr = windows_train(d, FIT_U, dyn)
    p = hp['ml']
    ml = MLBranch(model_type='xgboost', flatten_strategy=p['flatten'], random_state=SEED,
                  n_estimators=p['n_estimators'], max_depth=p['max_depth'], learning_rate=p['learning_rate'], **XGB_EXTRA)
    Ftr, names = ml_design(ml, Xtr, dyn, d, otr, stat)
    ml.model.fit(Ftr, ytr)
    # DL branch (BiLSTM-DA) with the W12 collapse guard
    Xtr_d, ytr_d, _ = windows_train(d, FIT_U, f_dl); Xva_d, yva_d, _ = windows_train(d, VAL_U, f_dl)
    const = rmse(yva_d, np.full_like(yva_d, ytr_d.mean()))
    for retry in range(MAX_DL_RETRIES + 1):
        tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED + 1000 * retry)
        dl, att = build_dual_attention_bilstm(window_size=W, n_features=len(f_dl), **hp['dl'])
        cb = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
              tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)]
        dl.fit(Xtr_d, ytr_d, epochs=EPOCHS, batch_size=BATCH, validation_data=(Xva_d, yva_d), callbacks=cb, verbose=0)
        if rmse(yva_d, dl.predict(Xva_d, batch_size=1024, verbose=0).ravel()) <= 0.9 * const:
            break
    M = dict(fset=fset, d=d, f_ml=f_ml, f_dl=f_dl, dyn=dyn, stat=stat, ml=ml, names=names, dl=dl, att=att, retries=retry)
    # Fusion weight and conformal residuals from the inner-validation engines only
    v = predict_windows(M, VAL_U)
    M['alpha'] = (1 / rmse(v.true, v.dl)) / (1 / rmse(v.true, v.ml) + 1 / rmse(v.true, v.dl))
    v['hybrid'] = M['alpha'] * v.dl + (1 - M['alpha']) * v.ml
    v['r'] = v.true - v.hybrid; v['st'] = stage_of(v.hybrid)
    M['resid'] = {st: g.r.to_numpy() for st, g in v.groupby('st', observed=True)}
    return M

def stage_of(p): return pd.cut(np.asarray(p, float), [-np.inf, 30, 80, np.inf], labels=['near', 'mid', 'early'])

def unit_windows(d, u, feats, cycles=None):
    # Windows ending at the given cycles (default: every cycle from W to end of life)
    g = d[d.unit == u].sort_values('cycle')
    v = g[feats].to_numpy(np.float32)
    X = sliding_window_view(v, (W, v.shape[1]))[:, 0]
    cyc = g['cycle'].to_numpy()[W - 1:]
    if cycles is not None:
        keep = np.isin(cyc, cycles); X, cyc = X[keep], cyc[keep]
    return X, cyc

def predict_windows(M, units, cycles_of=None):
    # ML, DL predictions for every window (or the given cycles per unit) of the given engines
    rows = []
    for u in units:
        cyc_sel = None if cycles_of is None else cycles_of.get(u)
        Xd, cyc = unit_windows(M['d'], u, M['f_dl'], cyc_sel)
        Xm, _ = unit_windows(M['d'], u, M['dyn'], cyc_sel)
        if len(cyc) == 0:
            continue
        F, _ = ml_design(M['ml'], Xm, M['dyn'], M['d'], [u] * len(Xm), M['stat'])
        rows.append(pd.DataFrame({'unit': u, 'cycle': cyc, 'ml': M['ml'].model.predict(F),
                                  'dl': M['dl'].predict(Xd, batch_size=1024, verbose=0).ravel(),
                                  'true': np.minimum(LIFE[u] - cyc, RUL_MAX),
                                  'true_uncapped': LIFE[u] - cyc}))
    return pd.concat(rows, ignore_index=True)

def add_uncertainty(M, df):
    df = df.copy()
    df['hybrid'] = M['alpha'] * df.dl + (1 - M['alpha']) * df.ml
    df['st'] = stage_of(df.hybrid)
    q = {st: np.quantile(np.abs(r), min(1, np.ceil((len(r) + 1) * (1 - ALPHA_CONF)) / len(r))) for st, r in M['resid'].items()}
    df['lo'] = np.clip(df.hybrid - df.st.map(q).astype(float), 0, None)
    df['hi'] = df.hybrid + df.st.map(q).astype(float)
    return df

def p_fail(M, pred, horizon):
    # P(RUL <= horizon) from the conformal residuals of the predicted stage
    r = M['resid'][stage_of([pred])[0]]
    return float(np.mean(pred + r <= horizon))

t0 = time.time()
MODELS = {}
for fset in CONFIGS:
    MODELS[fset] = fit_fold(fset)
    print(f"{fset:<9} trained | α = {MODELS[fset]['alpha']:.2f} | BiLSTM retries {MODELS[fset]['retries']} | "
          f"{len(MODELS[fset]['f_ml'])} ML / {len(MODELS[fset]['f_dl'])} DL features | {(time.time() - t0) / 60:.1f} min")

### 1.1 Check: accuracy and interval coverage on the test engines of the fold

In [ ]:
cuts_te = CUTS[CUTS.unit.isin(TE_U) & (CUTS.cut_cycle >= EVAL_MIN_HISTORY)]
cycles_of = cuts_te.groupby('unit').cut_cycle.apply(list).to_dict()
rows = []
TEST_PRED = {}
for fset, M in MODELS.items():
    p = add_uncertainty(M, predict_windows(M, TE_U, cycles_of))
    TEST_PRED[fset] = p
    rows.append({'config': fset, 'points': len(p), 'RMSE XGB': rmse(p.true, p.ml), 'RMSE DL': rmse(p.true, p.dl),
                 'RMSE hybrid': rmse(p.true, p.hybrid), 'coverage 90%': np.mean((p.true >= p.lo) & (p.true <= p.hi)),
                 'mean width': np.mean(p.hi - p.lo)})
    for s_, g in p.assign(subset=p.unit.map(train.groupby('unit').subset.first())).groupby('subset'):
        rows.append({'config': f"{fset} [{s_}]", 'points': len(g), 'RMSE XGB': rmse(g.true, g.ml), 'RMSE DL': rmse(g.true, g.dl),
                     'RMSE hybrid': rmse(g.true, g.hybrid), 'coverage 90%': np.mean((g.true >= g.lo) & (g.true <= g.hi)),
                     'mean width': np.mean(g.hi - g.lo)})
CHECK = pd.DataFrame(rows).round(3)
display(CHECK)
CHECK.to_csv(RESULTS / 'w15_fold_check.csv', index=False)

## 2. Engines and decision points

In [ ]:
# Two test engines of the fold: an FD001 engine with a life close to the FD001 median, and an FD003 engine
# with a long life (75th percentile of FD003), to show a wide-interval mid-life decision. Set ENGINES to override.
ENGINES = None
sub = train.groupby('unit').subset.first()
if ENGINES is None:
    te = pd.Series(LIFE[TE_U].values, index=TE_U)
    e1 = te[sub[TE_U] == 'FD001']; e3 = te[sub[TE_U] == 'FD003']
    ENGINES = [int((e1 - LIFE[sub == 'FD001'].median()).abs().idxmin()),
               int((e3 - LIFE[sub == 'FD003'].quantile(0.75)).abs().idxmin())]
print("Engines:", {u: f"{sub[u]}, life {LIFE[u]} cycles" for u in ENGINES})

TRAJ = {fset: add_uncertainty(M, predict_windows(M, ENGINES)) for fset, M in MODELS.items()}

# Decision points: engine 1 when the S+T hybrid first predicts RUL <= 60 (mid-life),
#                  engine 2 when it first predicts RUL <= 25 (near failure)
DECISIONS = []
for u, thr, label in [(ENGINES[0], 60, 'mid-life'), (ENGINES[1], 25, 'near failure')]:
    t = TRAJ[DEC_FSET][TRAJ[DEC_FSET].unit == u]
    hit = t[t.hybrid <= thr]
    row = hit.iloc[0] if len(hit) else t.iloc[-1]
    DECISIONS.append({'unit': u, 'cycle': int(row.cycle), 'label': label})
print("Decision points:", DECISIONS)

## 3. RUL trajectories with prediction intervals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
for ax, u in zip(axes, ENGINES):
    t = TRAJ[DEC_FSET][TRAJ[DEC_FSET].unit == u]; s = TRAJ[BASE_FSET][TRAJ[BASE_FSET].unit == u]
    ax.plot(t.cycle, t.true, 'k-', lw=1.5, label='True RUL (capped)')
    ax.fill_between(t.cycle, t.lo, t.hi, color='#9ecae1', alpha=0.5, label='90% interval (S+T)')
    ax.plot(t.cycle, t.hybrid, color='#08519c', lw=1.5, label='Hybrid, sensors + age')
    ax.plot(s.cycle, s.hybrid, color='#e6550d', lw=1, ls='--', label='Hybrid, sensors only')
    for dp in DECISIONS:
        if dp['unit'] == u:
            ax.axvline(dp['cycle'], color='grey', ls=':', lw=1.2)
            ax.text(dp['cycle'], RUL_MAX * 1.02, f"decision ({dp['label']})", ha='center', fontsize=8)
    ax.set_title(f"Engine {u} ({sub[u]}, life {LIFE[u]} cycles)")
    ax.set_xlabel('Cycle'); ax.set_ylabel('RUL (cycles)'); ax.set_ylim(0, RUL_MAX * 1.12)
axes[0].legend(fontsize=8, loc='lower left')
plt.tight_layout(); plt.savefig(RESULTS / 'fig_w15_rul_trajectories.png', dpi=200); plt.show()

## 4. SHAP explanations (XGBoost branch, all enterprise data)

In [ ]:
ME = MODELS[EXPL_FSET]
def ml_matrix(M, units, cycles_of=None):
    Fs, keys = [], []
    for u in units:
        cyc_sel = None if cycles_of is None else cycles_of.get(u)
        Xm, cyc = unit_windows(M['d'], u, M['dyn'], cyc_sel)
        if len(cyc) == 0:
            continue
        F, _ = ml_design(M['ml'], Xm, M['dyn'], M['d'], [u] * len(Xm), M['stat'])
        Fs.append(F); keys += [(u, c) for c in cyc]
    return np.vstack(Fs), keys

explainer = shap.TreeExplainer(ME['ml'].model)
F_te, keys_te = ml_matrix(ME, TE_U, cycles_of)
SV = explainer(F_te); SV.feature_names = ME['names']
orig = np.array([original_name(n) for n in ME['names']])
group = np.array([GROUP_OF.get(o, 'asset BI' if is_static(o) else 'sensor') for o in orig])

fig = plt.figure(figsize=(8, 6))
shap.plots.beeswarm(SV, max_display=15, show=False)
plt.title('SHAP values, XGBoost branch, all enterprise data (test truncation points)')
plt.tight_layout(); plt.savefig(RESULTS / 'fig_w15_shap_summary.png', dpi=200, bbox_inches='tight'); plt.show()

g_imp = pd.DataFrame(np.abs(SV.values), columns=group).T.groupby(level=0).sum().T.mean()
g_share = (g_imp / g_imp.sum()).sort_values(ascending=False)
print("Share of mean |SHAP| by group of variables:"); display(g_share.round(3))
g_share.to_csv(RESULTS / 'w15_shap_group_share.csv')

In [ ]:
# Waterfall at the first decision point, and contribution of each group along the trajectory of engine 1
u1, c1 = DECISIONS[0]['unit'], DECISIONS[0]['cycle']
F_u, keys_u = ml_matrix(ME, [u1])
SV_u = explainer(F_u); SV_u.feature_names = ME['names']
i_dec = [k[1] for k in keys_u].index(c1)
plt.figure()
shap.plots.waterfall(SV_u[i_dec], max_display=12, show=False)
plt.title(f"Engine {u1}, cycle {c1}: XGBoost prediction explained")
plt.tight_layout(); plt.savefig(RESULTS / 'fig_w15_shap_waterfall.png', dpi=200, bbox_inches='tight'); plt.show()

contrib = pd.DataFrame(SV_u.values, columns=group).T.groupby(level=0).sum().T
contrib['cycle'] = [k[1] for k in keys_u]
colors = {'sensor': '#6baed6', 'operating age': '#756bb1', 'operational BI': '#fd8d3c', 'asset BI': '#31a354'}
fig, ax = plt.subplots(figsize=(10, 4.5))
for gname in [g for g in colors if g in contrib]:
    ax.plot(contrib.cycle, contrib[gname], label=gname, color=colors[gname], lw=1.5)
ax.axhline(0, color='black', lw=0.7); ax.axvline(c1, color='grey', ls=':')
ax.set_xlabel('Cycle'); ax.set_ylabel('Sum of SHAP values (cycles of RUL)')
ax.set_title(f"Engine {u1}: contribution of each group of variables to the XGBoost prediction")
ax.legend(); plt.tight_layout(); plt.savefig(RESULTS / 'fig_w15_shap_evolution.png', dpi=200); plt.show()
contrib.to_csv(RESULTS / 'w15_shap_evolution_engine1.csv', index=False)

## 5. Attention (BiLSTM, all enterprise data)

In [ ]:
Xd_u, cyc_u = unit_windows(ME['d'], u1, ME['f_dl'])
aw = ME['att'].predict(Xd_u, batch_size=512, verbose=0)
FW, TW = np.asarray(aw['feature_weights']), np.asarray(aw['temporal_weights'])   # (n, W, f), (n, W)
j = list(cyc_u).index(c1)

fig, axes = plt.subplots(1, 3, figsize=(20, 5), gridspec_kw={'width_ratios': [2.2, 1, 2]})
im = axes[0].imshow(FW[j].T, aspect='auto', cmap='viridis')
axes[0].set_yticks(range(len(ME['f_dl']))); axes[0].set_yticklabels(ME['f_dl'], fontsize=7)
axes[0].set_xlabel('Time step in the window (oldest → newest)')
axes[0].set_title(f'Feature attention, engine {u1}, cycle {c1}'); plt.colorbar(im, ax=axes[0])
axes[1].bar(range(W), TW[j], color='#3182bd'); axes[1].set_xlabel('Time step'); axes[1].set_title('Temporal attention')
mean_fw = pd.DataFrame(FW.mean(axis=1), columns=ME['f_dl'])
top = mean_fw.mean().sort_values(ascending=False).index[:6]
for c in top:
    axes[2].plot(cyc_u, mean_fw[c], label=c)
axes[2].axvline(c1, color='grey', ls=':'); axes[2].set_xlabel('Cycle'); axes[2].set_ylabel('Mean feature attention')
axes[2].set_title('Evolution of feature attention (top 6 variables)'); axes[2].legend(fontsize=7)
plt.tight_layout(); plt.savefig(RESULTS / 'fig_w15_attention.png', dpi=200); plt.show()
mean_fw.assign(cycle=cyc_u).to_csv(RESULTS / 'w15_attention_evolution_engine1.csv', index=False)

## 6. Decision scenarios

Risk-aware cost model on top of `ScenarioGenerator`, which provides the candidate PM times and the cost of each intervention from the enterprise context (PM and corrective costs, downtime penalty, production priority, shift, maintenance window, technician and spare-parts availability, contract penalty).

- Failure probability of a PM planned Δ cycles ahead: p = P(RUL < Δ), from the conformal residuals of the predicted life stage (point-estimate policy: p = 1 if the predicted RUL < Δ, else 0).
- Expected direct cost: (1 − p)·(PM intervention + PM downtime) + p·(corrective intervention + corrective downtime).
- **Criterion: expected cost per cycle of operation**, expected cost / E[min(RUL, Δ)] (the classical age-replacement criterion). Planning the PM too early wastes useful life (few cycles for the cost); too late risks a corrective intervention. "Do nothing" = corrective cost / E[RUL].
- The generator's production-loss term (revenue × cycles until PM), which always favours the earliest PM, is not used.

In [ ]:
RAW = DATA['real']           # unscaled values of the enterprise variables
def context_at(u, cycle, pred, typical=False):
    r = RAW[(RAW.unit == u) & (RAW.cycle == cycle)].iloc[0] if not typical else TYPICAL
    cat = lambda v: int(np.argmax([r.get(f"{v}_{k}", 0) for k in range(3)]))
    return UnitContext(unit_id=int(u), current_cycle=int(cycle), predicted_rul=float(max(pred, 1)),
                       pm_cost=r.pm_cost, cm_cost=r.cm_cost, downtime_penalty=r.downtime_penalty,
                       revenue_per_hour=r.revenue_per_hour, technician_available=int(round(r.technician_available)),
                       spare_parts_available=int(round(r.spare_parts_available)), spare_parts_lead_time=r.spare_parts_lead_time,
                       labor_rate_standard=r.labor_rate_standard, labor_rate_overtime=r.labor_rate_overtime,
                       contract_penalty_active=int(round(r.contract_penalty_active)),
                       production_priority=cat('production_priority'), maintenance_window=int(round(r.maintenance_window)),
                       shift_pattern=cat('shift_pattern'))

# Typical context: fleet medians (continuous) and most frequent values (binary, categorical), training engines only
tr_raw = RAW[RAW.unit.isin(TR_U)]
TYPICAL = tr_raw[OP_COLS].median()
for c in OP_COLS:
    if tr_raw[c].nunique() <= 2:
        TYPICAL[c] = tr_raw[c].mode().iloc[0]

class RiskAwareGenerator(ScenarioGenerator):
    # samples: draws of the remaining life (cycles) given the prediction; point estimate = one draw
    def __init__(self, samples, **kw):
        super().__init__(**kw); self.T = np.clip(np.asarray(samples, float), 0, None)
    def generate_scenarios(self, ctx, **kw):
        kw.setdefault('n_pm_times', 15); kw.setdefault('restoration_levels', [0.9])
        sc = super().generate_scenarios(ctx, **kw)
        dn = next(s for s in sc if s.action == 'do_nothing')
        cm = dn.intervention_cost + dn.downtime_cost
        for s in sc:
            s.production_loss_cost = 0.0
            if s.action == 'do_nothing':
                s.failure_probability, s.total_cost = 1.0, cm
                s.score = cm / max(self.T.mean(), 1.0)
                continue
            dt = s.intervention_time - ctx.current_cycle
            p = float(np.mean(self.T < dt))
            s.failure_probability = p
            s.total_cost = (1 - p) * (s.intervention_cost + s.downtime_cost) + p * cm
            s.score = s.total_cost / max(np.mean(np.minimum(self.T, dt)), 1.0)     # expected cost per cycle
        return sorted(sc, key=lambda s: s.score)

def decide(policy, u, cycle, preds):
    # preds: dict fset -> trajectory/test prediction table; returns (best scenario, all scenarios, context, pred)
    fset = BASE_FSET if policy == 'sensors only' else DEC_FSET
    M = MODELS[fset]
    row = preds[fset][(preds[fset].unit == u) & (preds[fset].cycle == cycle)].iloc[0]
    pred = float(row.hybrid)
    ctx = context_at(u, cycle, pred, typical=(policy == 'no context'))
    samples = [pred] if policy == 'point estimate' else pred + M['resid'][stage_of([pred])[0]]
    sc = RiskAwareGenerator(samples).generate_scenarios(ctx)
    return sc[0], sc, ctx, row

POLICIES = ['full', 'no context', 'point estimate', 'sensors only']
rows = []
for dp in DECISIONS:
    u, c = dp['unit'], dp['cycle']
    for pol in POLICIES:
        best, sc, ctx, row = decide(pol, u, c, TRAJ)
        dt = best.intervention_time - c if best.action != 'do_nothing' else np.nan
        rows.append({'engine': u, 'situation': dp['label'], 'cycle': c, 'policy': pol,
                     'predicted RUL': round(row.hybrid, 1), '90% interval': f"[{row.lo:.0f}, {row.hi:.0f}]",
                     'true RUL': int(LIFE[u] - c),
                     'P(fail ≤ 15)': round(p_fail(MODELS[BASE_FSET if pol == 'sensors only' else DEC_FSET], row.hybrid, 15), 2),
                     'P(fail ≤ 30)': round(p_fail(MODELS[BASE_FSET if pol == 'sensors only' else DEC_FSET], row.hybrid, 30), 2),
                     'recommendation': best.name, 'PM in (cycles)': dt,
                     'expected cost': round(best.total_cost), 'cost per cycle': round(best.score, 1),
                     'risk of failure before PM': round(best.failure_probability, 3),
                     'priority': ctx.priority_label, 'technician': ctx.technician_available,
                     'spare parts': ctx.spare_parts_available, 'window': ctx.maintenance_window,
                     'fails before PM (truth)': bool(LIFE[u] - c < dt) if not np.isnan(dt) else True})
DEC = pd.DataFrame(rows)
DEC.to_csv(RESULTS / 'w15_decision_scenarios.csv', index=False)
display(DEC)

for dp in DECISIONS:
    best, sc, ctx, row = decide('full', dp['unit'], dp['cycle'], TRAJ)
    print('\n' + '=' * 80)
    print(sg.generate_recommendation(sc, ctx))
    display(sg.cost_benefit_table(sc).head(8))
    sg.plot_scenario_comparison(sc, ctx, save_path=str(RESULTS / f"fig_w15_scenarios_engine{dp['unit']}.png"))
    plt.show()

## 7. Fleet-level evaluation of the decisions

Each policy decides at every scored truncation point of the 40 test engines of the fold. Outcomes are computed from the true failure time: a PM planned after the true failure is a failure (corrective cost); otherwise the PM cost is paid and the remaining life at the PM is unused. The summary criterion is the realised cost per operated cycle (total cost / total cycles operated until the intervention). Realised costs always use the **actual** enterprise context, whatever context the policy used to decide.

In [ ]:
gen_actual = ScenarioGenerator()
rows = []
for _, cpt in cuts_te.iterrows():
    u, c = int(cpt.unit), int(cpt.cut_cycle)
    true_unc = int(LIFE[u] - c)
    for pol in POLICIES:
        best, sc, ctx, row = decide(pol, u, c, TEST_PRED)
        actual = context_at(u, c, float(row.hybrid))
        cm = gen_actual._do_nothing_scenario(actual)
        cm_cost = cm.intervention_cost + cm.downtime_cost
        if best.action == 'do_nothing':
            failed, unused, cost, operated = True, 0, cm_cost, true_unc
        else:
            dt = best.intervention_time - c
            failed = true_unc < dt
            pm = gen_actual._pm_scenario(actual, best.intervention_time, best.restoration_level)
            cost = cm_cost if failed else pm.intervention_cost + pm.downtime_cost
            unused = 0 if failed else true_unc - dt
            operated = true_unc if failed else dt
        rows.append({'unit': u, 'cycle': c, 'policy': pol, 'failed': failed, 'unused life': unused,
                     'realised cost': cost, 'cycles operated': operated, 'true RUL': true_unc})
FLEET = pd.DataFrame(rows)
POL = FLEET.groupby('policy').agg(points=('unit', 'size'), failure_rate=('failed', 'mean'),
                                  mean_unused_life=('unused life', 'mean'), mean_cost=('realised cost', 'mean'),
                                  total_cost=('realised cost', 'sum'), cycles=('cycles operated', 'sum')).loc[POLICIES]
POL['cost per operated cycle'] = POL.total_cost / POL.cycles
POL['cost rate vs full (%)'] = 100 * (POL['cost per operated cycle'] / POL.loc['full', 'cost per operated cycle'] - 1)
display(POL.round(3))

# Same comparison for FD001 and FD003 engines separately
FLEET['subset'] = FLEET.unit.map(train.groupby('unit').subset.first())
POL_SUB = FLEET.groupby(['subset', 'policy']).agg(points=('unit', 'size'), failure_rate=('failed', 'mean'),
                                                   mean_unused_life=('unused life', 'mean'),
                                                   total_cost=('realised cost', 'sum'), cycles=('cycles operated', 'sum'))
POL_SUB['cost per operated cycle'] = POL_SUB.total_cost / POL_SUB.cycles
POL_SUB = POL_SUB.round(3)
display(POL_SUB)
POL_SUB.to_csv(RESULTS / 'w15_policy_comparison_by_subset.csv')
POL.to_csv(RESULTS / 'w15_policy_comparison.csv')
FLEET.to_csv(RESULTS / 'w15_policy_points.csv', index=False)

# Paired comparison of realised cost per point: full vs each other policy
FLEET['cost per cycle'] = FLEET['realised cost'] / FLEET['cycles operated'].clip(lower=1)
piv = FLEET.pivot_table(index=['unit', 'cycle'], columns='policy', values='cost per cycle')
for pol in POLICIES[1:]:
    d = piv[pol] - piv['full']
    p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    print(f"{pol:<15} − full: mean Δ cost per operated cycle {d.mean():8.1f} (two-sided Wilcoxon p = {p:.3g})")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col, title in zip(axes, ['failure_rate', 'mean_unused_life', 'cost per operated cycle'],
                          ['Failures before the planned PM', 'Unused life at PM (cycles)', 'Realised cost per operated cycle']):
    ax.bar(POL.index, POL[col], color=['#08519c', '#fd8d3c', '#969696', '#e6550d'])
    ax.set_title(title); ax.tick_params(axis='x', rotation=20)
plt.tight_layout(); plt.savefig(RESULTS / 'fig_w15_policy_comparison.png', dpi=200); plt.show()

## 7b. Decision dashboard figure for the article

A compact, print-oriented summary of one engine at its decision point, in two widths: **single column** (3.5 in, panels stacked) and **double column** (7.16 in, panels side by side). Serif font, two colours (prediction in blue, failure risk in orange; palette checked for colour-blind separation), grey for the enterprise context, no decorative elements. Saved as PDF (vector, for LaTeX) and PNG.

In [ ]:
import matplotlib as mpl
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
BLUE, ORANGE, INK, MUTED = '#2b5d9b', '#c0582a', '#222222', '#6b6b6b'
STYLE = {'font.family': 'serif', 'font.serif': ['Times New Roman', 'Nimbus Roman', 'STIXGeneral', 'DejaVu Serif'],
         'mathtext.fontset': 'stix', 'font.size': 7.5, 'axes.titlesize': 7.5, 'axes.labelsize': 7.5,
         'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 6.5, 'axes.linewidth': 0.6,
         'axes.edgecolor': '#444444', 'axes.spines.top': False, 'axes.spines.right': False,
         'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
         'axes.grid': True, 'axes.grid.axis': 'y', 'grid.color': '#e6e6e6', 'grid.linewidth': 0.5,
         'legend.frameon': False, 'savefig.bbox': 'tight', 'savefig.pad_inches': 0.02}

def dashboard_data(dp):
    u, c = dp['unit'], dp['cycle']
    t = TRAJ[DEC_FSET][TRAJ[DEC_FSET].unit == u].copy()
    M = MODELS[DEC_FSET]
    t['risk15'] = [p_fail(M, h, 15) for h in t.hybrid]
    t['risk30'] = [p_fail(M, h, 30) for h in t.hybrid]
    raw = RAW[RAW.unit == u].set_index('cycle').loc[t.cycle]
    ctx = pd.DataFrame({'Priority': np.argmax(raw[[f'production_priority_{k}' for k in range(3)]].to_numpy(), 1) / 2,
                        'Technician': raw.technician_available.round().to_numpy(),
                        'Spare parts': raw.spare_parts_available.round().to_numpy(),
                        'Maint. window': raw.maintenance_window.round().to_numpy()}, index=t.cycle)
    best, sc, cx, row = decide('full', u, c, TRAJ)
    dn = next(s for s in sc if s.action == 'do_nothing')
    truth = t[['cycle', 'true']].copy()          # full true trajectory, shown in hindsight only
    t, ctx = t[t.cycle <= c], ctx[ctx.index <= c]  # the dashboard shows what is known at the decision cycle
    rec = dict(cycle=c, pred=row.hybrid, lo=row.lo, hi=row.hi, r30=p_fail(M, row.hybrid, 30),
               pm=None if best.action == 'do_nothing' else best.intervention_time, p=best.failure_probability,
               saving=100 * (1 - best.score / dn.score), ctx=cx, truth=truth)
    rec['xmax'] = max(c, rec['pm'] or c) + 10
    return t, ctx, rec

def panel_rul(ax, t, rec, legend=True):
    ax.fill_between(t.cycle, t.lo, t.hi, color=BLUE, alpha=0.15, lw=0, label='90% interval')
    ax.plot(t.cycle, t.hybrid, color=BLUE, lw=1.1, label='Predicted RUL')
    tr = rec['truth'][rec['truth'].cycle <= rec['xmax']]
    ax.plot(tr.cycle, tr.true, color=MUTED, lw=0.8, ls=(0, (3, 2)), label='True RUL (hindsight)')
    ax.errorbar([rec['cycle']], [rec['pred']], yerr=[[rec['pred'] - rec['lo']], [rec['hi'] - rec['pred']]],
                fmt='o', ms=3, color=BLUE, elinewidth=0.9, capsize=2, zorder=4)
    ax.set_xlim(t.cycle.min(), rec['xmax'])
    ax.axvline(rec['cycle'], color=MUTED, lw=0.6)
    ax.annotate('now', (rec['cycle'], RUL_MAX * 1.04), ha='center', va='bottom',
                fontsize=6.5, color=MUTED, annotation_clip=False)
    if rec['pm'] is not None:
        ax.plot([rec['pm']], [0], marker='v', ms=4.5, color=INK, clip_on=False, zorder=5)
        ax.annotate('planned PM', (rec['pm'], 0), xytext=(0, 7), textcoords='offset points', ha='center',
                    fontsize=6.5, color=INK)
    ax.set_ylim(0, RUL_MAX * 1.04); ax.set_ylabel('RUL (cycles)')
    if legend:
        ax.legend(loc='lower left', handlelength=1.6)

def panel_risk(ax, t, rec):
    ax.plot(t.cycle, t.risk15, color=ORANGE, lw=1.1, label='within 15 cycles')
    ax.plot(t.cycle, t.risk30, color=ORANGE, lw=1.1, ls=(0, (3, 2)), label='within 30 cycles')
    ax.axvline(rec['cycle'], color=MUTED, lw=0.6)
    ax.set_ylim(0, 1.02); ax.set_yticks([0, 0.5, 1]); ax.set_ylabel('Failure risk')
    ax.set_xlim(t.cycle.min(), rec['xmax'])
    ax.legend(loc='upper left', handlelength=1.6)

def panel_context(ax, ctx, rec):
    rows = list(ctx.columns)
    img = ctx.T.to_numpy(float)
    ax.imshow(img, aspect='auto', cmap=mpl.colors.LinearSegmentedColormap.from_list('ctx', ['#f2f2f2', '#8a8a8a']),
              vmin=0, vmax=1, interpolation='nearest',
              extent=[ctx.index.min() - 0.5, ctx.index.max() + 0.5, len(rows) - 0.5, -0.5])
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows); ax.grid(False)
    ax.tick_params(axis='y', length=0)
    for s in ('left', 'bottom'):
        ax.spines[s].set_visible(False)
    ax.axvline(rec['cycle'], color=BLUE, lw=0.8)
    ax.set_xlabel('Cycle')
    ax.set_xlim(right=rec['xmax'])
    ax.text(1.0, 1.02, 'grey = available / higher priority', transform=ax.transAxes, ha='right', va='bottom',
            fontsize=6, color=MUTED)

def rec_text(rec):
    cx = rec['ctx']
    lines = [f"Cycle {rec['cycle']}: RUL {rec['pred']:.0f} [{rec['lo']:.0f}, {rec['hi']:.0f}] cycles",
             f"Risk of failure within 30 cycles: {rec['r30']:.0%}"]
    if rec['pm'] is None:
        lines.append('Recommendation: continue monitoring')
    else:
        lines.append(f"Recommendation: PM at cycle {rec['pm']} (in {rec['pm'] - rec['cycle']} cycles), "
                     f"risk before PM {rec['p']:.0%}")
        lines.append(f"Expected cost per cycle: {rec['saving']:.0f}% below run-to-failure")
    lines.append(f"Context: {cx.priority_label.lower()} priority, technician {'available' if cx.technician_available else 'unavailable'}, "
                 f"parts {'in stock' if cx.spare_parts_available else 'on order'}, "
                 f"{'maintenance window open' if cx.maintenance_window else 'no maintenance window'}")
    return lines

for dp in DECISIONS:
    t, ctx, rec = dashboard_data(dp)
    u = dp['unit']
    with mpl.rc_context(STYLE):
        # Single column: 3.5 in wide, stacked panels sharing the cycle axis
        fig = plt.figure(figsize=(3.5, 4.4))
        gs = fig.add_gridspec(3, 1, height_ratios=[2.1, 1.0, 0.95], hspace=0.12)
        a1 = fig.add_subplot(gs[0]); a2 = fig.add_subplot(gs[1], sharex=a1); a3 = fig.add_subplot(gs[2], sharex=a1)
        panel_rul(a1, t, rec); panel_risk(a2, t, rec); panel_context(a3, ctx, rec)
        plt.setp(a1.get_xticklabels(), visible=False); plt.setp(a2.get_xticklabels(), visible=False)
        a1.set_title(f"Engine {u} ({sub[u]})", loc='left', fontsize=7.5)
        fig.text(0.0, -0.01, '\n'.join(rec_text(rec)), ha='left', va='top', fontsize=6.5, color=INK, linespacing=1.35)
        for ext in ('pdf', 'png'):
            fig.savefig(RESULTS / f'fig_w15_dashboard_single_engine{u}.{ext}', dpi=300)
        plt.show()

        # Double column: 7.16 in wide, three panels side by side + recommendation
        fig = plt.figure(figsize=(7.16, 2.3))
        gs = fig.add_gridspec(2, 3, width_ratios=[1.3, 1.15, 0.85], height_ratios=[1, 0.9], wspace=0.5, hspace=0.3)
        a1 = fig.add_subplot(gs[:, 0]); a2 = fig.add_subplot(gs[0, 1]); a3 = fig.add_subplot(gs[1, 1], sharex=a2)
        panel_rul(a1, t, rec); a1.set_xlabel('Cycle'); a1.set_title(f"Engine {u} ({sub[u]})", loc='left')
        panel_risk(a2, t, rec); plt.setp(a2.get_xticklabels(), visible=False)
        panel_context(a3, ctx, rec)
        a4 = fig.add_subplot(gs[:, 2]); a4.axis('off')
        a4.text(0, 1, 'Decision support', fontsize=7.5, fontweight='bold', va='top', color=INK)
        a4.text(0, 0.86, '\n'.join(rec_text(rec)).replace(', parts', ',\nparts').replace('Context: ', 'Context:\n'),
                va='top', fontsize=6.5, color=INK, linespacing=1.45, wrap=True)
        for ext in ('pdf', 'png'):
            fig.savefig(RESULTS / f'fig_w15_dashboard_double_engine{u}.{ext}', dpi=300)
        plt.show()

## 8. Dashboard data and save

In [ ]:
# One table per engine for the decision dashboard: prediction, interval, risk, context, recommendation
dash = []
for u in ENGINES:
    t = TRAJ[DEC_FSET][TRAJ[DEC_FSET].unit == u]
    for _, row in t.iloc[::5].iterrows():
        best, sc, ctx, _ = decide('full', u, int(row.cycle), TRAJ)
        dash.append({'unit': u, 'cycle': int(row.cycle), 'true RUL': row.true, 'predicted RUL': row.hybrid,
                     'lower 90%': row.lo, 'upper 90%': row.hi,
                     'P(fail ≤ 15)': p_fail(MODELS[DEC_FSET], row.hybrid, 15), 'P(fail ≤ 30)': p_fail(MODELS[DEC_FSET], row.hybrid, 30),
                     'recommendation': best.name, 'expected cost': best.total_cost, 'cost per cycle': best.score, 'priority': ctx.priority_label,
                     'technician': ctx.technician_available, 'spare parts': ctx.spare_parts_available,
                     'maintenance window': ctx.maintenance_window})
DASH = pd.DataFrame(dash).round(3)
DASH.to_csv(RESULTS / 'w15_dashboard_data.csv', index=False)
display(DASH.head(10))

if SAVE_DIR is not None:
    for f in RESULTS.iterdir():
        if f.is_file() and f.name.startswith(('w15', 'fig_w15')):
            shutil.copy(f, SAVE_DIR / f.name)
    print(f"Results copied to {SAVE_DIR}")

## 9. Reading the results (for Sections 5.6–5.7)

- **Fold check (1.1):** the fold RMSEs should be close to the W12 values for seed 42; coverage close to 0.90.
- **SHAP (4):** the group shares quantify how much of the XGBoost prediction rests on sensors, operating age, operational BI and asset history. The evolution plot shows when the age dominates (early life) and when sensors take over (near failure).
- **Attention (5):** feature and temporal attention at the decision point; attention is less stable than SHAP across folds (W12), so use it for individual explanations.
- **Decisions (6):** compare the four policies at each decision point: how the enterprise context changes the timing and cost, and how uncertainty changes the urgency. Check whether the recommended PM would have preceded the true failure.
- **FD001 vs FD003:** the fold check and the fleet evaluation are also reported per subset; FD003 engines (two failure modes, longer and more variable lives) are where the operating age helped most in W12.
- **Fleet (7):** the operational test of the framework: failure rate, unused life and realised cost of each policy. The cost model is the one of `scenario_generator.py` and its parameters are illustrative; report relative differences, not absolute amounts.